# 08 — End-to-End Gating Verification

**Purpose:** Verify that `src/gating.py` correctly selects sessions and produces results
consistent with the retrospective simulation in notebook 07.

**Approach:**
1. Auto-detect existing full-run results (BGL + HDFS normalized JSONL)
2. If results exist: load margins + signatures, build mock objects, call `gate()` directly
3. If results don't exist: run the pipeline end-to-end with gating
4. Verify K, ordering, signature coverage at all budget levels
5. Cross-check against notebook 07 simulation results

**Key insight:** `gate()` only uses `margin` from `ScreenerOutput`. When full-run
results exist, no re-run of screener, BM25, or LLM is needed.

## 1. Imports & Configuration

In [9]:
import sys
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# Project root
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Verify imports
from src.gating import GatingMode, GatingConfig, gate
from src.screener import ScreenerOutput
from src.data_loader import Session

print(f"Project root: {PROJECT_ROOT}")
print(f"[OK] All imports successful")
print(f"[OK] GatingMode values: {[m.value for m in GatingMode]}")

Project root: /home/dave/agentic-log-explanations
[OK] All imports successful
[OK] GatingMode values: ['explain_all', 'top_k']


In [10]:
# === Auto-detect and load existing full-run results ===

# Known result files (add new runs here)
RESULT_FILES = {
    'BGL':  PROJECT_ROOT / "results" / "explanations_BGL_20260213_160034.normalized.jsonl",
    'HDFS': PROJECT_ROOT / "results_HDFS" / "explanations_HDFS_20260215_220648.normalized.jsonl",
}

# Simulation results for cross-check
SIM_PATH = PROJECT_ROOT / "results" / "gating_simulation_results.json"

def load_results(path: Path, dataset: str):
    """Load normalized JSONL into (records, sessions, screener_outputs)."""
    records, sess_list, so_list = [], [], []
    with open(path, 'r') as f:
        for line in f:
            d = json.loads(line)
            if d.get('explanation') is None:
                continue
            sid = d['session_id']
            margin = d['screener']['margin']
            prob = d['screener']['prob']
            sig_name = d.get('normalized_signature',
                             d['explanation'].get('signature', {}).get('name', 'UNKNOWN'))
            records.append({
                'session_id': sid,
                'margin': margin,
                'uncertainty': 1.0 - margin,
                'normalized_signature': sig_name,
                'total_tokens': d['metrics']['total_tokens'],
            })
            sess_list.append(Session(session_id=sid, split="test", lines=[], label=d['label']))
            so_list.append(ScreenerOutput(
                session_id=sid, pred=1, logits=[0.0, 0.0], prob=prob, margin=margin,
            ))
    df = pd.DataFrame(records)
    return df, sess_list, so_list

# Load all available datasets
datasets = {}
for ds_name, path in RESULT_FILES.items():
    if path.exists():
        df_ds, sess_ds, so_ds = load_results(path, ds_name)
        datasets[ds_name] = {'df': df_ds, 'sessions': sess_ds, 'screener_outputs': so_ds}
        print(f"[OK] {ds_name}: loaded {len(df_ds)} sessions, "
              f"{df_ds['normalized_signature'].nunique()} signatures")
    else:
        print(f"[SKIP] {ds_name}: {path.name} not found")

print(f"\nDatasets available for gating verification: {list(datasets.keys())}")

[OK] BGL: loaded 6290 sessions, 56 signatures
[OK] HDFS: loaded 2527 sessions, 13 signatures

Datasets available for gating verification: ['BGL', 'HDFS']


## 2. Mode a — Explain-All (Pass-Through)

Verify `gate()` with `GatingMode.EXPLAIN_ALL` returns all anomaly sessions unchanged for each dataset.

In [11]:
# === Mode a: Explain-All — should return everything ===
config_a = GatingConfig(mode=GatingMode.EXPLAIN_ALL, budget=1.0)

for ds_name, ds in datasets.items():
    result_a = gate(ds['sessions'], ds['screener_outputs'], config_a)
    n_a = len(result_a)
    assert n_a == len(ds['sessions']), f"{ds_name}: Mode a should return all {len(ds['sessions'])}, got {n_a}"

    # Build sid-to-signature lookup and store back
    ds['sid_to_sig'] = dict(zip(ds['df']['session_id'], ds['df']['normalized_signature']))
    ds['all_sigs'] = set(ds['df']['normalized_signature'].unique())

    print(f"[OK] {ds_name}: Mode a returned all {n_a} sessions, "
          f"{len(ds['all_sigs'])} unique signatures")

[OK] BGL: Mode a returned all 6290 sessions, 56 unique signatures
[OK] HDFS: Mode a returned all 2527 sessions, 13 unique signatures


## 3. Mode b — Top-K by Uncertainty (All Budget Levels)

Verify `gate()` with `GatingMode.TOP_K` at all budget levels for each dataset.

In [12]:
# === Mode b: Top-K at multiple budget levels, all datasets ===
BUDGET_LEVELS = [0.10, 0.20, 0.30, 0.50, 0.75, 1.00]

all_gate_results = {}

for ds_name, ds in datasets.items():
    N = len(ds['sessions'])
    print(f"\n{'='*65}")
    print(f"{ds_name} — Total anomaly sessions: {N}")
    print(f"{'='*65}")
    print(f"{'Budget':>8} {'K':>6} {'Returned':>10} {'Sigs':>6} {'Coverage':>10} {'Max Margin':>12}")
    print("-" * 60)

    gate_results = {}
    for B in BUDGET_LEVELS:
        config = GatingConfig(mode=GatingMode.TOP_K, budget=B)
        result = gate(ds['sessions'], ds['screener_outputs'], config)
        K_expected = max(1, math.floor(B * N))

        selected_sigs = set(ds['sid_to_sig'][s.session_id] for s, _ in result)
        coverage = len(selected_sigs) / len(ds['all_sigs'])
        max_margin = max(so.margin for _, so in result)

        gate_results[B] = {
            'K': K_expected, 'returned': len(result),
            'sigs': len(selected_sigs), 'coverage': coverage,
            'max_margin': max_margin, 'selected_sigs': selected_sigs,
        }
        ok = len(result) == K_expected
        print(f"{B:>8.0%} {K_expected:>6} {len(result):>10} {len(selected_sigs):>6} "
              f"{coverage:>9.1%} {max_margin:>12.6f}  {'[OK]' if ok else '[FAIL]'}")

    all_gate_results[ds_name] = gate_results


BGL — Total anomaly sessions: 6290
  Budget      K   Returned   Sigs   Coverage   Max Margin
------------------------------------------------------------
     10%    629        629     42     75.0%     1.000000  [OK]
     20%   1258       1258     53     94.6%     1.000000  [OK]
     30%   1887       1887     54     96.4%     1.000000  [OK]
     50%   3145       3145     55     98.2%     1.000000  [OK]
     75%   4717       4717     56    100.0%     1.000000  [OK]
    100%   6290       6290     56    100.0%     1.000000  [OK]

HDFS — Total anomaly sessions: 2527
  Budget      K   Returned   Sigs   Coverage   Max Margin
------------------------------------------------------------
     10%    252        252     10     76.9%     0.999998  [OK]
     20%    505        505     12     92.3%     0.999998  [OK]
     30%    758        758     12     92.3%     0.999998  [OK]
     50%   1263       1263     12     92.3%     0.999998  [OK]
     75%   1895       1895     12     92.3%     0.999998  [

## 4. Ordering Verification

Verify that `gate()` selects sessions sorted by ascending margin (= descending uncertainty) for each dataset.

In [13]:
# === Verify ordering at B=0.20 for all datasets ===

for ds_name, ds in datasets.items():
    config_b20 = GatingConfig(mode=GatingMode.TOP_K, budget=0.20)
    result_b20 = gate(ds['sessions'], ds['screener_outputs'], config_b20)

    margins_selected = [so.margin for _, so in result_b20]
    is_sorted = all(margins_selected[i] <= margins_selected[i+1]
                     for i in range(len(margins_selected)-1))

    # Verify these are truly the K smallest margins
    all_margins = sorted(so.margin for so in ds['screener_outputs'])
    K = len(result_b20)
    threshold = all_margins[K - 1]
    max_sel = max(margins_selected)

    print(f"\n--- {ds_name} (B=0.20, K={K}) ---")
    print(f"  [{'OK' if is_sorted else 'FAIL'}] Ascending margin order")
    print(f"  [{'OK' if max_sel <= threshold + 1e-10 else 'FAIL'}] "
          f"All selected <= threshold ({threshold:.6f})")
    print(f"  Sample (5 most uncertain):")
    for s, so in result_b20[:5]:
        sig = ds['sid_to_sig'][s.session_id]
        print(f"    {s.session_id[:25]:<25}  margin={so.margin:.6f}  sig={sig}")


--- BGL (B=0.20, K=1258) ---
  [OK] Ascending margin order
  [OK] All selected <= threshold (1.000000)
  Sample (5 most uncertain):
    BGL_03594190               margin=0.008237  sig=APP__CIOD_STREAM_ERROR
    BGL_03298260               margin=0.035584  sig=KERNEL__DATA_TLB_ERROR
    BGL_00074760               margin=0.043008  sig=KERNEL__DATA_TLB_ERROR
    BGL_01510380               margin=0.057411  sig=KERNEL__MISSING_OR_INVALID_FIELDS
    BGL_04056860               margin=0.062017  sig=LINKCARD__MONITOR_FAILURE

--- HDFS (B=0.20, K=505) ---
  [OK] Ascending margin order
  [OK] All selected <= threshold (0.999998)
  Sample (5 most uncertain):
    HDFS_blk_5648741609066690  margin=0.145755  sig=DATANODE__BLOCK_VERIFICATION_FAILED
    HDFS_blk_666218916297487   margin=0.944964  sig=DATANODE__BLOCK_VERIFICATION_FAILED
    HDFS_blk_2960911979907752  margin=0.951486  sig=DATANODE__BLOCK_VERIFICATION_FAILED
    HDFS_blk_8956532717568639  margin=0.987518  sig=DATANODE__BLOCK_VERIFICATION_

## 5. Cross-Check with Notebook 07 Simulation

Compare `gate()` results with the retrospective simulation from `gating_simulation_results.json`.
Both should produce identical signature coverage at each budget level for both datasets.

In [14]:
# === Cross-check with notebook 07 retrospective simulation ===
sim_results = pd.read_json(SIM_PATH)

overall_ok = True

for ds_name in datasets:
    sim_ds = sim_results[
        (sim_results['dataset'] == ds_name) &
        (sim_results['strategy'] == 'Uncertainty')
    ].set_index('budget')

    gate_res = all_gate_results[ds_name]
    N = len(datasets[ds_name]['sessions'])
    total_sigs = len(datasets[ds_name]['all_sigs'])

    print(f"\n{'='*70}")
    print(f"{ds_name} — gate() vs Notebook 07 Simulation (Uncertainty strategy)")
    print(f"{'='*70}")
    print(f"{'Budget':>8} {'gate()':>12} {'Sim':>12} {'gate() Sigs':>12} {'Sim Sigs':>10} {'Match':>7}")
    print("-" * 70)

    ds_match = True
    for B in BUDGET_LEVELS:
        g_cov = gate_res[B]['coverage']
        g_sigs = gate_res[B]['sigs']

        if B in sim_ds.index:
            s_cov = sim_ds.loc[B, 'coverage']
            s_sigs = int(sim_ds.loc[B, 'unique_sigs'])
            match = abs(g_cov - s_cov) < 1e-6
            if not match:
                ds_match = False
                overall_ok = False
            print(f"{B:>8.0%} {g_cov:>11.1%} {s_cov:>11.1%} "
                  f"{g_sigs:>12} {s_sigs:>10} {'[OK]' if match else '[DIFF]':>7}")
        else:
            print(f"{B:>8.0%} {g_cov:>11.1%} {'N/A':>12} {g_sigs:>12} {'N/A':>10}")

    # Operating point
    g20 = gate_res[0.20]
    missed = datasets[ds_name]['all_sigs'] - g20['selected_sigs']
    print(f"\n  B=0.20: {g20['returned']}/{N} sessions, "
          f"{g20['coverage']:.1%} coverage ({g20['sigs']}/{total_sigs})")
    if missed:
        print(f"  Missing: {sorted(missed)}")

    status = "[OK]" if ds_match else "[FAIL]"
    print(f"  {status} {ds_name} gate() matches simulation")

print(f"\n{'='*70}")
if overall_ok:
    print("[OK] ALL DATASETS VERIFIED: gate() is consistent with retrospective simulation")
else:
    print("[WARN] Some datasets have mismatches -- investigate")


BGL — gate() vs Notebook 07 Simulation (Uncertainty strategy)
  Budget       gate()          Sim  gate() Sigs   Sim Sigs   Match
----------------------------------------------------------------------
     10%       75.0%       75.0%           42         42    [OK]
     20%       94.6%       94.6%           53         53    [OK]
     30%       96.4%          N/A           54        N/A
     50%       98.2%       98.2%           55         55    [OK]
     75%      100.0%      100.0%           56         56    [OK]
    100%      100.0%      100.0%           56         56    [OK]

  B=0.20: 1258/6290 sessions, 94.6% coverage (53/56)
  Missing: ['KERNEL__INFO', 'KERNEL__MICROLOADER_ASSERTION', 'KERNEL__RECEIVED_SIGNAL']
  [OK] BGL gate() matches simulation

HDFS — gate() vs Notebook 07 Simulation (Uncertainty strategy)
  Budget       gate()          Sim  gate() Sigs   Sim Sigs   Match
----------------------------------------------------------------------
     10%       76.9%       76.9%   